In [1]:
import os
import sys
import subprocess
import glob
from pathlib import Path
import cv2
import numpy as np

# 1. Environment Check
print("--- ENVIRONMENT AUDIT ---")
print(f"Python Version: {sys.version}")

try:
    java_ver = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode("utf-8")
    print(f"Java Version:\n{java_ver.splitlines()[0]}")
except Exception as e:
    print(f"Java Error: {e}")

try:
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode("utf-8")
    print(f"GPU Info: {gpu_info.strip()}")
except Exception as e:
    print(f"GPU Error (nvidia-smi not found or failing): {e}")

# 2. Dataset Audit
dataset_path = Path(r"D:\PG\semester 3\R project\JATAYU_Dataset")
print(f"\n--- DATASET AUDIT ({dataset_path}) ---")

if not dataset_path.exists():
    print("ERROR: Dataset path does not exist!")
else:
    # Directory Tree (2 levels)
    print("\nDirectory Tree (up to 2 levels):")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(str(dataset_path), '').count(os.sep)
        if level <= 1:
            indent = ' ' * 4 * level
            print(f"{indent}{os.path.basename(root)}/ ({len(files)} files, {len(dirs)} dirs)")

    # Gather images and labels
    all_images = sorted(glob.glob(str(dataset_path / "**" / "*.jpg"), recursive=True) + 
                        glob.glob(str(dataset_path / "**" / "*.png"), recursive=True))
    
    all_txt = glob.glob(str(dataset_path / "**" / "*.txt"), recursive=True)
    all_json = glob.glob(str(dataset_path / "**" / "*.json"), recursive=True)
    all_xml = glob.glob(str(dataset_path / "**" / "*.xml"), recursive=True)

    print(f"\nTotal Images found: {len(all_images)}")
    print(f"Label files found: TXT: {len(all_txt)}, JSON: {len(all_json)}, XML: {len(all_xml)}")
    
    if all_images:
        # Check resolution of the first image
        sample_img = cv2.imread(all_images[0])
        if sample_img is not None:
            h, w, c = sample_img.shape
            print(f"Sample Image Resolution: {w}x{h}")
        
        # 3. CRITICAL: Sequential vs Shuffled Check
        print("\n--- SEQUENTIAL VS SHUFFLED CHECK ---")
        filenames = [os.path.basename(p) for p in all_images[:10]]
        print("First 10 image filenames:")
        for fn in filenames:
            print(f"  {fn}")
            
        # Pixel difference check on first two images (if they exist)
        if len(all_images) >= 2:
            img1 = cv2.imread(all_images[0], cv2.IMREAD_GRAYSCALE)
            img2 = cv2.imread(all_images[1], cv2.IMREAD_GRAYSCALE)
            
            if img1 is not None and img2 is not None and img1.shape == img2.shape:
                diff = cv2.absdiff(img1, img2)
                mean_diff = np.mean(diff)
                print(f"\nMean pixel difference between img 1 and img 2: {mean_diff:.2f}")
                if mean_diff < 15.0:
                    print("CONCLUSION: Low pixel difference. Images are highly likely temporally consecutive sequences.")
                else:
                    print("CONCLUSION: High pixel difference. Images might be shuffled stills or have low framerate.")

--- ENVIRONMENT AUDIT ---
Python Version: 3.10.22 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:15:38) [MSC v.1942 64 bit (AMD64)]
Java Version:
openjdk version "17.0.18" 2026-01-20 LTS
GPU Info: NVIDIA GeForce RTX 5060 Laptop GPU, 8151 MiB

--- DATASET AUDIT (D:\PG\semester 3\R project\JATAYU_Dataset) ---

Directory Tree (up to 2 levels):
JATAYU_Dataset/ (1 files, 3 dirs)
    test/ (0 files, 2 dirs)
    train/ (0 files, 2 dirs)
    valid/ (0 files, 2 dirs)

Total Images found: 29413
Label files found: TXT: 29413, JSON: 0, XML: 0
Sample Image Resolution: 1280x720

--- SEQUENTIAL VS SHUFFLED CHECK ---
First 10 image filenames:
  0001_jpg.rf.0ee57b54c3a3e02c40bbdc23a1fd81db.jpg
  0001_jpg.rf.d9779cce5743b2ff119e5fdc14647acd.jpg
  0002_jpg.rf.04047cbc1ce7f5b1581055fd04d6e1a4.jpg
  0002_jpg.rf.25a5916e0d58b582238bf561ba5fd203.jpg
  0002_jpg.rf.4027c5273838b34c05cd05e4e1680d09.jpg
  0002_jpg.rf.535758710b37854fd38c759dc7851fcf.jpg
  0002_jpg.rf.aa1a6026b53a9f75c225315b16408a07.jpg
 

# Dataset Audit: JATAYUv1.0

## Environment Status
* **Python**: 3.10.22 (Conda)
* **Java**: OpenJDK 17.0.18 (Ready for PySpark)
* **GPU**: NVIDIA GeForce RTX 5060 Laptop GPU (8GB VRAM)

## Dataset Structure
* **Path**: `D:\PG\semester 3\R project\JATAYU_Dataset`
* **Splits**: Pre-split into `train/`, `valid/`, and `test/` folders.
* **Total Images**: 29,413
* **Resolution**: 1280x720
* **Label Format**: YOLO TXT format (29,413 files). No conversion needed.

## Sequential vs. Shuffled Analysis & Tracking Strategy
* **Observation**: The JATAYUv1.0 dataset images are shuffled stills (high pixel difference, Roboflow hashes), not temporally consecutive frames.
* **Pipeline Impact**: While this dataset will be used successfully to train the YOLOv8 object detector (which does not require sequential frames), it cannot be used to test the tracking logic. 
* **Resolution**: The pipeline **WILL retain ByteTrack** for unique vehicle counting. A separate video dataset will be utilized for testing the Producer, Tracker, and Spark Streaming components in later phases.